In [1]:
import tensorflow as tf
# tf.logging.set_verbosity(tf.logging.ERROR)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

We shall use the ImageDataGenerator class to feed in the training and validation data to the model. This class can also be used to generate augmented data.

To know more about ImageDataGenerator class, visit https://keras.io/preprocessing/image/#imagedatagenerator-class

In [2]:
train_datagenerator = ImageDataGenerator(rescale=1./255)
test_datagenerator = ImageDataGenerator(rescale=1./255)

train_datagenerator = train_datagenerator.flow_from_directory(
    r"C:\Nandeesh\krishnaik_ml_course\training_images\train",
    target_size=(128,128),
    batch_size=10,
    class_mode='binary')

test_datagenerator = test_datagenerator.flow_from_directory(
    r"C:\Nandeesh\krishnaik_ml_course\training_images\validation",
    target_size=(128,128),
    batch_size=3,
    class_mode='binary')

Found 513 images belonging to 2 classes.
Found 66 images belonging to 2 classes.


Our model will have 3 Convolution2D layers. You can increse or decrease as per your needs.

In [3]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(32, (3,3),padding='same', activation='relu', input_shape=(128,128,3)),
    tf.keras.layers.MaxPooling2D((2,2),2),
    
    tf.keras.layers.Conv2D(64, (3,3), padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2),2),     
     
    tf.keras.layers.Conv2D(128, (3,3), padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2),2),   
    
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(512, activation='relu'),
    
    tf.keras.layers.Dense(1, activation='sigmoid')
])

c:\Nandeesh\git\kn_ml_course\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [4]:
print(model.summary())

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 32768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │    16,777,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,871,489 (64.36 MB)

 Trainable params: 16,871,489 (64.36 MB)

 Non-trainable params: 0 (0.00 B)

None


We shall use Adam optimizer with a learning rate of 0.001 (hyperparameter). We choose 'binary_crossentropy' loss as our model is a binary calssifier (i.e, we have only 2 classes)

In [5]:
model.compile(loss='binary_crossentropy',
             optimizer=tf.keras.optimizers.Adam(0.001),
             metrics=['accuracy'])

At the end of each epoch we can check if the model has reached the required accuracy and terminate the training.

In [6]:
DESIRED_ACCURACY = 0.85

class myCallback(tf.keras.callbacks.Callback):
  def on_epoch_end(self, epoch, logs={}):
    if((logs.get('acc')>DESIRED_ACCURACY) and (logs.get('val_acc')>DESIRED_ACCURACY )):
      print("\nReached 85% accuracy so cancelling training!")
      self.model.stop_training = True

callbacks = myCallback()

In [6]:
DESIRED_ACCURACY = 0.9

class myCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epochs, logs={}) :
        if(logs.get('acc') is not None and logs.get('acc') >= DESIRED_ACCURACY) :
            print('\nReached 99.9% accuracy so cancelling training!')
            self.model.stop_training = True

callbacks = myCallback()

To know more about fit_generator visit https://keras.io/models/model/#fit_generator

In [7]:
model.fit(
    train_datagenerator,
    epochs=20,
    validation_data = test_datagenerator,
    callbacks = [callbacks]
    )

Epoch 1/20


c:\Nandeesh\git\kn_ml_course\venv\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


52/52 ━━━━━━━━━━━━━━━━━━━━ 24s 399ms/step - accuracy: 0.5694 - loss: 1.0844 - val_accuracy: 0.8030 - val_loss: 0.4529
Epoch 2/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 26s 461ms/step - accuracy: 0.8969 - loss: 0.2545 - val_accuracy: 0.7879 - val_loss: 0.4730
Epoch 3/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 26s 462ms/step - accuracy: 0.9600 - loss: 0.1542 - val_accuracy: 0.7727 - val_loss: 0.4759
Epoch 4/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 26s 463ms/step - accuracy: 0.9400 - loss: 0.2058 - val_accuracy: 0.8182 - val_loss: 0.4407
Epoch 5/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 27s 476ms/step - accuracy: 0.9699 - loss: 0.0952 - val_accuracy: 0.8333 - val_loss: 0.4006
Epoch 6/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 26s 463ms/step - accuracy: 0.9649 - loss: 0.1027 - val_accuracy: 0.7879 - val_loss: 0.6600
Epoch 7/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 26s 460ms/step - accuracy: 0.9662 - loss: 0.0785 - val_accuracy: 0.8485 - val_loss: 0.4331
Epoch 8/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 26s 459ms/step - accuracy: 0.9744 - loss: 0.0585 - val_accuracy: 0.818

In [8]:
model.save('mymodel_large.h5')